(rd.tunecell)=
# 7. Tune Cell

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

## Introduction

In order to adjust the tunes of the ring, the phase advance per FODO cell in one of the straight
sections can be adjusted, while ensuring that it is still matched to the periodic solution in each
of the surrounding arcs. We'll put this "tune cell" in the 2 o'clock straight section, between the
sextants 1 and 3, and name its quadrupoles by appending `_2` to the names of the quadrupoles they
replace.

```{figure} figures/tunecell.svg
:width: 100%

The tune cell in the 2 o'clock straight section.
```

Required for the match is:

1. Setting the phase advances in the `fodossf_2`/`fodossr_2` cells of the straight section for the
   desired tunes $Q_1 = 54.08$ and $Q_2 = 54.14$ (2 constraints).
2. Matching from the dispersion suppressor to the new periodic Twiss parameters of the
   `fodossf_2`/`fodossr_2` cells (4 constraints).
3. Matching the `fodossf_2`/`fodossr_2` cells to the dispersion creator (4 more constraints, but
   these are already met by the mirror symmetry).

These 6 constraints can be met using the four quadrupoles of the forward matching section
(`qff2_2`, `qdf2_2`, `qff3_2`, and `qdf3_2`) and the two quadrupoles of the straight section cells
(`qfss_2` and `qdss_2`).

## Example: Matching the Tune Cell

1. Start with the ring with the IR of [chapter 6](interaction-region.md):

In [2]:
from lattices.ch06_interaction_region import *
sb.twiss(ring, at=[])  # The tunes before

Twiss:
summ:
 q1     = 54.029705863767695
 q2     = 54.12406173420637
 etac   = 0.0006460281777786715
 alphac = 0.000646028997649545

df:
0×26 DataFrame
 Row │ index  name  kind  s    beta1  phi1  dx   x    beta2  phi2  dy   y    slip  alpha1  alpha2  ⋯
     │                    [m]  [m]    [2π]  [m]  [m]  [m]    [2π]  [m]  [m]  [m]   [1]     [1]     ⋯
─────┴──────────────────────────────────────────────────────────────────────────────────────────────
                                                                                  11 columns omitted

2. Define the new quadrupoles, starting with the strengths of those they replace:

In [3]:
with sb.elements():
    qfss_2 = sb.Quadrupole(L=0.5, Kn1=qfss.Kn1)
    qdss_2 = sb.Quadrupole(L=0.5, Kn1=qdss.Kn1)
    qff2_2 = sb.Quadrupole(L=0.5, Kn1=qff2.Kn1)
    qdf2_2 = sb.Quadrupole(L=0.5, Kn1=qdf2.Kn1)
    qff3_2 = sb.Quadrupole(L=0.5, Kn1=qff3.Kn1)
    qdf3_2 = sb.Quadrupole(L=0.5, Kn1=qdf3.Kn1)

3. Define the lines of the tune cell, and put them into the sextants 1 and 3. The dispersion
   suppressor and the dispersion creator keep their quadrupoles:

In [4]:
fodossf_2 = [qfss_2, d1, db, d2, qdss_2, d1, db, d2]
fodossr_2 = [qfss_2, d2, db, d1, qdss_2, d2, db, d1]
mssf_2 = [qff2_2, d1, db, d2, qdf2_2, d1, db, d2, qff3_2, d1, db, d2, qdf3_2, d1, db, d2]
mdcr_2 = [qfss_2, d2, db, d1, qdf3_2, d2, db, d1, qff3_2, d2, db, d1, qdf2_2, d2, db, d1]
dispcrer_2 = [qff2_2, d2, bh, d1, qdf1, d2, bh, d1, qff1, d2, bh, d1, qd, d2, bh, d1]
arc_to_ssf_2 = dispsupf + mssf_2
ss_to_arcr_2 = mdcr_2 + dispcrer_2

sextant1 = fodossf*4 + ss_to_arcf + fodoaf*20 + arc_to_ssf_2 + fodossf_2*4
sextant3 = fodossr_2*4 + ss_to_arcr_2 + fodoar*20 + arc_to_ssr + fodossr*4

ring = sb.Beamline(sextant1 + sextant3 + sextant5 + sextant7 + sextant9 + sextant11,
                   species_ref=electron, pc_ref=pc_ref)

4. The targets need three `TwissExpr`s: of the new straight section cell (its periodic Twiss
   parameters), of the line from the arc into the new straight section (the first half of the tune
   cell), and of the ring (its tunes):

In [5]:
cell_ss2 = sb.Beamline(fodossf_2, species_ref=electron, pc_ref=pc_ref)
line_2 = sb.Beamline(arc_to_ssf_2, species_ref=electron, pc_ref=pc_ref)

tw_ss2 = sb.TwissExpr(cell_ss2)
tw_2 = sb.TwissExpr(line_2, a_initial=tw_af.a[-1])
tw_ring = sb.TwissExpr(ring)

   All three depend on the knobs, and {jl}`match!`{py}`sb.match` computes each of them once per
   evaluation.

5. Run the match:

In [6]:
sb.match(vary=[sb.Knob(q, "Kn1") for q in (qfss_2, qdss_2, qff2_2, qdf2_2, qff3_2, qdf3_2)],
         targets=[tw_2.beta1[-1] == tw_ss2.beta1[sb.begin],
                  tw_2.alpha1[-1] == tw_ss2.alpha1[sb.begin],
                  tw_2.beta2[-1] == tw_ss2.beta2[sb.begin],
                  tw_2.alpha2[-1] == tw_ss2.alpha2[sb.begin],
                  tw_ring.q1 == 54.08, tw_ring.q2 == 54.14])

Match converged (all targets met): 3 iterations, 4 evaluations, 0.724 s
 Target                          value             goal      met
 beta1[end] - beta1[qfss_2#1]    2.312816605e-12   == 0.0    yes
 alpha1[end] - alpha1[qfss_2#1]  -5.138112158e-13  == 0.0    yes
 beta2[end] - beta2[qfss_2#1]    5.329070518e-14   == 0.0    yes
 alpha2[end] - alpha2[qfss_2#1]  1.501576641e-13   == 0.0    yes
 q1                              54.08             == 54.08  yes
 q2                              54.14             == 54.14  yes
 Knob        before         after          lower  upper
 qfss_2.Kn1  0.3519574526   0.3569268647   -Inf   Inf
 qdss_2.Kn1  -0.3519574526  -0.3543728037  -Inf   Inf
 qff2_2.Kn1  0.3568781026   0.356879803    -Inf   Inf
 qdf2_2.Kn1  -0.3527483412  -0.3530781971  -Inf   Inf
 qff3_2.Kn1  0.3788662921   0.3812929182   -Inf   Inf
 qdf3_2.Kn1  -0.3612538547  -0.3627590489  -Inf   Inf


   The tunes of the ring are now 54.08 and 54.14, and the beta functions are only changed in the
   tune cell:

In [7]:
tw = sb.twiss(ring)
from matplotlib.figure import Figure
fig = Figure(figsize=(8, 3), layout="constrained")
ax = fig.subplots()
rows = slice(0.0, 1300.0)  # The sextants 1 and 3
ax.plot(tw.s[rows], tw.beta1[rows], label="β₁")
ax.plot(tw.s[rows], tw.beta2[rows], label="β₂")
ax.set_xlabel("s [m]")
ax.set_ylabel("β [m]")
ax.legend(loc="upper right")
fig

<Figure size 800x300 with 1 Axes>

## Exercises

1. **Periodicity inside the tune cell:** In the Bmad version of this tutorial, the periodicity of
   the Twiss parameters inside the tune cell was a target of its own: the Twiss parameters at the
   fourth `qfss_2` of the ring must be equal to those at the fifth. Write these targets with
   `tw_ring` only. Hint: `ring[[qfss_2]]` is the vector of all placements of `qfss_2` in the
   ring, and each of them is a location by itself.

2. **Other tunes:** Change the tunes to 54.12 and 54.18. How much do the quadrupole strengths of the
   tune cell change? Which tunes can the tune cell reach?

The lattice file of this chapter is
[`lattices/ch07_tune_cell.py`](https://github.com/bmad-sim/SciBmad.jl/tree/main/docs/src/ring-design/lattices/ch07_tune_cell.py).